In [ ]:
import sys;sys.path.append("../..")

import pickle

import numpy as np
import pandas as pd
from pathlib import Path

from pipeline.utils.ranking_utils import (
    FULL_ARMS, K_FOLDS, K_PLACEBO, SEED, cv_folds, load_frame, make_arms, placebo_frame,
)
from pipeline.utils.eval_utils import normalized_positions, per_query_meanpos

DATA_DIR = Path("../../datasets/findhr")
JUDGE = "qwen3-8b"     # edit to switch judge
JUDGE_DIR = DATA_DIR / "judges" / JUDGE

In [ ]:
print(f"judge: {JUDGE}")
MODEL_DIR = JUDGE_DIR / "model"

ARMS = make_arms()
BASELINE = "B_only"
COMPARE_ARMS = [a for a in FULL_ARMS if a != BASELINE]
GRADES = np.array([0, 1, 2])

In [ ]:
df = load_frame(JUDGE)

In [ ]:
FOLDS = list(cv_folds(df, k=K_FOLDS, seed=SEED))

cached = pickle.loads((MODEL_DIR / "runs.pkl").read_bytes())

models = {
    arm: [m.booster_ for m in cached["results"][arm]["models"]] for arm in ARMS if arm != "Placebo"
}

# Sorted by seed: cell 4 pairs placebo_models[s] with placebo_frame(df, s) positionally.
placebo_models = [
    [m.booster_ for m in r["models"]] for r in sorted(cached["placebo_runs"], key=lambda r: r["seed"])
]
assert len(placebo_models) == K_PLACEBO, "runs.pkl holds a different number of placebo replicates"

print(f"{len(models)} real arms x {K_FOLDS} folds + {K_PLACEBO} placebo replicates x {K_FOLDS} folds")

In [ ]:
# Positions are within-query, so each fold is scored independently by its own models and the
# blocks are concatenated in fold order. `placebo_p` is stacked the same way, which is what
# keeps its rows aligned with `pos`.
pcols = ARMS["Placebo"]
blocks, placebo_blocks = [], []

for i, parts in enumerate(FOLDS):

    tsub, tg = parts["test"]
    block = pd.DataFrame({
        "id_j": tsub["id_j"].values,
        "id_c": tsub["id_c"].values,
        "relevance": tsub["relevance"].values,
        "n": tsub.groupby("id_j")["id_c"].transform("size").values
    })

    for arm in models.keys():
        block[f"p_{arm}"] = normalized_positions(models[arm][i].predict(tsub[ARMS[arm]]), tg)
    blocks.append(block)

    # placebo_frame(df, s) rewrites only the placebo column, so .loc[tsub.index] picks exactly
    # the rows this fold's replicate-s model was tested on, in the same order.
    placebo_blocks.append(
        np.column_stack(
            [
                normalized_positions(placebo_models[s][i].predict(placebo_frame(df, s).loc[tsub.index, pcols]), tg)
                for s in range(K_PLACEBO)
            ]
        )
    )

pos = pd.concat(blocks, ignore_index=True)

# Placebo replicates kept as an (N, K) matrix, not collapsed yet: a statistic must be computed
# inside each replicate and averaged afterwards.
placebo_p = np.vstack(placebo_blocks)  # (N, K), rows in the same fold order as `pos`

assert len(pos) == len(placebo_p) == len(df), "a fold went missing"
assert pos["id_j"].nunique() == df["id_j"].nunique() and not pos.duplicated(["id_j", "id_c"]).any()

In [ ]:
RESULTS_DIR = JUDGE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

positions = pos.assign(**{f"p_placebo_{s:02d}": placebo_p[:, s] for s in range(K_PLACEBO)})
positions.to_csv(RESULTS_DIR / "positions.csv", index=False, float_format="%.8f")
print(f"wrote results/positions.csv | {positions.shape}")

In [ ]:
for arm in [a for a in ARMS if a != "Placebo"]:
    pos[f"d_{arm}"] = pos[f"p_{BASELINE}"] - pos[f"p_{arm}"]

placebo_d = pos[f"p_{BASELINE}"].values[:, None] - placebo_p  # (N, K)
pos["p_Placebo"] = placebo_p.mean(axis=1)
pos["d_Placebo"] = placebo_d.mean(axis=1)

In [ ]:
DESERVING = 2
g2 = (pos["relevance"] == DESERVING).values

base = per_query_meanpos(pos, pos[f"p_{BASELINE}"].values)
# One mean Δ per placebo replicate, computed inside each replicate, averaged afterwards.
# Descriptive: it fills the Placebo row below, nothing is tested against it.
placebo_stat = np.array([float(np.mean(base - per_query_meanpos(pos, p))) for p in placebo_p.T])

rows = []
for arm in COMPARE_ARMS:
    if arm == "Placebo":
        # Placebo row: median over the K replicates, not over the queries.
        rows.append({"arm": f"Placebo (K={K_PLACEBO})", "n_queries": len(base),
                     "mean_Δ": round(float(placebo_stat.mean()), 4),
                     "median_Δ": round(float(np.median(placebo_stat)), 4),
                     "win_frac": round(float((placebo_d[g2] > 0).mean()), 3)})
        continue
    d = (base - per_query_meanpos(pos, pos[f"p_{arm}"].values)).values
    rows.append({"arm": arm, "n_queries": len(d),
                 "mean_Δ": round(float(d.mean()), 4),
                 "median_Δ": round(float(np.median(d)), 4),
                 "win_frac": round(float((d > 0).mean()), 3)})

In [ ]:
headline = pd.DataFrame(rows).set_index("arm")
print(f"Placebo reference (K={K_PLACEBO} replicates, descriptive): mean Δ "
      f"mean={placebo_stat.mean():+.4f} sd={placebo_stat.std(ddof=1):.4f} "
      f"max={placebo_stat.max():+.4f}\n")
print(headline.to_string())